In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler




In [ ]:
folder_path = "/content/drive/MyDrive/data_kaggle/alzheimers_disease_data"

x = np.load(f"{folder_path}/X.npy")
y = np.load(f"{folder_path}/y.npy")

In [ ]:
print(x.shape, y.shape)
print(np.unique(y, return_counts=True))

(2149, 35) (2149,)
(array([0, 1]), array([1389,  760]))


In [ ]:
features= ['Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI',
       'Smoking', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality',
       'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease',
       'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP',
       'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL',
       'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment',
       'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion',
       'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks',
       'Forgetfulness']

train_test split

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# تقسیم داده‌ها: 80٪ آموزش، 20٪ تست
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.6, random_state=42, stratify=y)

# مدل درخت تصمیم
dt_model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=4, random_state=42)
dt_model.fit(x_train, y_train)

# پیش‌بینی
y_pred = dt_model.predict(x_test)

# محاسبه متریک‌ها
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print("Confusion Matrix:\n", cm)


Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
Confusion Matrix:
 [[834   0]
 [  0 456]]


PCA Scatter

In [ ]:
from scipy.spatial import ConvexHull
from matplotlib.patches import Ellipse

pca = PCA(n_components=2)
x_pca = pca.fit_transform(x)  # اینجا X ماتریس فیچرهاست

def plot_confidence_ellipse(x, y, ax, n_std=2.0, facecolor='none', **kwargs):
    cov = np.cov(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    eigenvals, eigenvecs = np.linalg.eigh(cov)
    order = eigenvals.argsort()[::-1]
    eigenvals, eigenvecs = eigenvals[order], eigenvecs[:, order]

    angle = np.degrees(np.arctan2(*eigenvecs[:,0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigenvals)

    ellipse = Ellipse(xy=(mean_x, mean_y), width=width, height=height,
                      angle=angle, facecolor=facecolor, **kwargs)
    ax.add_patch(ellipse)

plt.figure(figsize=(8,6))
ax = plt.gca()

colors = ['blue', 'red']
for class_value, color in zip(np.unique(y), colors):
    points = x_pca[y == class_value]
    plt.scatter(points[:,0], points[:,1], c=color, alpha=0.6, label=f"Class {class_value}")
    plot_confidence_ellipse(points[:,0], points[:,1], ax, n_std=3, edgecolor=color)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA - 2D Projection with Class Ellipses')
plt.legend()
plt.show()

Feature *Importance*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import pandas as pd

# مدل روی داده‌های اصلی
rf = RandomForestClassifier(random_state=1)
rf.fit(x, y)  # X = 35 فیچر، y = کلاس‌ها

# اهمیت فیچرها
importances = pd.Series(rf.feature_importances_, index=features)
importances.sort_values().plot(kind='barh', figsize=(7,5))
plt.title('Feature Importances')
plt.show()